# HW-2: Detector comparison
All detectors use the same protocol: frames from `prep/train` built by `hw-2.ipynb`, the same image size where possible, and one COCO evaluator on the original val frames.

Run `hw-2.ipynb` up to the training folder first, then connect this notebook to the same Colab runtime. Results are saved to Drive after every run, so finished runs are skipped when a cell runs again. Run one thing at a time, because the runtime has one GPU.

Start with `SMOKE_TEST = True`: every model trains briefly and is scored on 12 val frames, in a separate `runs-smoke` folder, so errors appear in minutes. The numbers mean nothing. Then set it to `False` and restart the kernel.

The flags work as in the project notebook:
- `TRAIN = False`: never train, only score weights that already exist.
- `FORCE = True`: score finished runs again without retraining them.
- `ONLY = ['frcnn-base']`: run only the named runs.

An interrupted YOLO or RT-DETR run continues from `last.pt` with `resume=True`. A torchvision run saves `model.pt` only at the end, so an interrupted one starts again.

In [ ]:
%pip install -q ultralytics pycocotools kagglehub

import kagglehub
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

SMOKE_TEST = True
TRAIN = True
FORCE = False
ONLY = []

COMPETITION = 'multiple-object-video-detection-hw-2-ucu-ai-2026'
RUNS = Path('/content/drive/MyDrive/ai-ucu-hw-2') / ('runs-smoke' if SMOKE_TEST else 'runs')
RUNS.mkdir(parents=True, exist_ok=True)
PREP = Path('/content/prep')
DATA = Path(kagglehub.competition_download(COMPETITION))

assert (PREP / 'data.yaml').exists(), 'Run hw-2.ipynb up to the training folder first'

In [ ]:
import json
import math
import time
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.io import read_image
from torchvision.models.detection import (fasterrcnn_resnet50_fpn_v2, retinanet, retinanet_resnet50_fpn_v2,
                                          roi_heads)
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.ops import generalized_box_iou_loss
from tqdm.auto import tqdm
from ultralytics import RTDETR, YOLO

IMGSZ = 960
EPOCHS = 1 if SMOKE_TEST else 6
BATCH = 4
LR = 0.005
MAX_DET = 500
SCORE_THRESHOLD = 0.01
SEED = 42

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS = RUNS / 'model_results.csv'
TRAIN_LIMIT, VAL_LIMIT = (16, 12) if SMOKE_TEST else (None, None)

## Shared evaluator
`COCOeval` gives mAP, AP50, AP75 and AP by object size with the official definitions. `AP_small` covers boxes under 32x32 px in the original frame.

Ignored regions stay in the ground truth as `iscrowd=1`, so detections inside them are not penalised. That matches the competition, so val frames are used as they are, without masking. The dataset stores each region once per class because `COCOeval` works class by class. `maxDets` is raised to 500 to match the limit on the leaderboard.

In [ ]:
val_coco = json.loads((DATA / 'val' / 'annotations_coco.json').read_text())
val_frames = [(im['id'], DATA / 'val' / im['file_name']) for im in val_coco['images']][:VAL_LIMIT]


def coco_metrics(rows):
    ground_truth = COCO(str(DATA / 'val' / 'annotations_coco.json'))
    evaluation = COCOeval(ground_truth, ground_truth.loadRes(rows), 'bbox')
    evaluation.params.maxDets = [1, 100, MAX_DET]
    evaluation.params.imgIds = [image_id for image_id, _ in val_frames]
    evaluation.evaluate()
    evaluation.accumulate()
    evaluation.summarize()
    names = ['mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large']
    return dict(zip(names, evaluation.stats[:6].tolist()))


def save_result(name, family, settings, metrics):
    table = pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame()
    row = {'run': name, 'family': family, 'settings': settings, **metrics}
    table = pd.concat([table[table.run != name] if len(table) else table, pd.DataFrame([row])], ignore_index=True)
    table.to_csv(RESULTS, index=False)
    return row


def is_done(name):
    return RESULTS.exists() and name in set(pd.read_csv(RESULTS).run)


def should_run(name):
    return (not ONLY or name in ONLY) and (FORCE or not is_done(name))


def unfinished(last):
    return torch.load(last, map_location='cpu', weights_only=False).get('optimizer') is not None

## Ultralytics models on the shared evaluator
`ultralytics_rows` turns predictions into COCO rows, so YOLO and RT-DETR are scored exactly like the torchvision models. COCO `category_id` is 1-based, so the class id is shifted by one.

In [ ]:
def ultralytics_rows(model, imgsz):
    ids = {str(path): image_id for image_id, path in val_frames}
    rows, seconds = [], 0.0
    results = model.predict(list(ids), stream=True, imgsz=imgsz, conf=SCORE_THRESHOLD, max_det=MAX_DET, verbose=False)
    for result in tqdm(results, total=len(ids)):
        seconds += result.speed['inference'] / 1000
        for (x1, y1, x2, y2), score, cls in zip(result.boxes.xyxy.tolist(), result.boxes.conf.tolist(),
                                                result.boxes.cls.int().tolist()):
            rows.append({'image_id': ids[result.path], 'category_id': cls + 1, 'score': score,
                         'bbox': [x1, y1, x2 - x1, y2 - y1]})
    return rows, 1000 * seconds / len(ids)


def run_ultralytics(name, family, settings, make_model, imgsz=IMGSZ, batch=8, epochs=20, **train_args):
    if not should_run(name):
        return
    Model = RTDETR if family == 'query-based' else YOLO
    last, best = RUNS / name / 'weights' / 'last.pt', RUNS / name / 'weights' / 'best.pt'
    if SMOKE_TEST:
        epochs, train_args = 1, {**train_args, 'fraction': 0.01}
    start = time.time()
    if TRAIN and not last.exists():
        make_model().train(data=str(PREP / 'data.yaml'), imgsz=imgsz, epochs=epochs, batch=batch, seed=SEED,
                           project=str(RUNS), name=name, exist_ok=True, plots=False, **train_args)
    elif TRAIN and unfinished(last):
        Model(str(last)).train(resume=True)
    if not last.exists() or unfinished(last):
        print('Not trained yet, skipping:', name)
        return
    minutes = (time.time() - start) / 60
    rows, ms = ultralytics_rows(Model(best), imgsz)
    print(save_result(name, family, settings, {**coco_metrics(rows), 'ms_per_frame': ms, 'train_minutes': minutes}))


## Torchvision training
Faster R-CNN and RetinaNet share one loop: pretrained COCO weights, SGD with warmup and a cosine schedule, mixed precision, and a horizontal flip. Images are read from `prep/train`, so ignore regions are masked as in the YOLO runs.

The default limits are too low for this data: Faster R-CNN keeps 100 detections per image and RetinaNet keeps 300, while one train frame has 180 objects and `SCORE_THRESHOLD` keeps uncertain boxes for mAP. Both are set to `MAX_DET`.

In [ ]:
class FrameDataset(Dataset):
    def __init__(self, limit=None):
        self.images = sorted((PREP / 'train' / 'images').rglob('*.jpg'))[:limit]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        path = self.images[index]
        image = read_image(str(path)).float() / 255
        height, width = image.shape[1:]
        label_path = Path(str(path).replace('/images/', '/labels/')).with_suffix('.txt')
        labels = np.loadtxt(label_path, ndmin=2).reshape(-1, 5)
        cx, cy, w, h = (labels[:, 1:] * [width, height, width, height]).T
        boxes = torch.tensor(np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1), dtype=torch.float32)
        if torch.rand(1) < 0.5:
            image = image.flip(2)
            boxes[:, [0, 2]] = width - boxes[:, [2, 0]]
        return image, {'boxes': boxes, 'labels': torch.tensor(labels[:, 0] + 1, dtype=torch.int64)}


def collate(batch):
    return tuple(zip(*batch))

In [ ]:
def giou_fastrcnn_loss(class_logits, box_regression, labels, regression_targets):
    labels = torch.cat(labels)
    targets = torch.cat(regression_targets)
    classification = nn.functional.cross_entropy(class_logits, labels)
    positive = torch.where(labels > 0)[0]
    deltas = box_regression.reshape(len(labels), -1, 4)[positive, labels[positive]]

    def to_box(delta):
        dx, dy, dw, dh = (delta / delta.new_tensor([10, 10, 5, 5])).unbind(1)
        w, h = dw.clamp(max=math.log(1000 / 16)).exp(), dh.clamp(max=math.log(1000 / 16)).exp()
        return torch.stack([dx - w / 2, dy - h / 2, dx + w / 2, dy + h / 2], dim=1)

    box = generalized_box_iou_loss(to_box(deltas), to_box(targets[positive]), reduction='sum')
    return classification, box / labels.numel()


original_fastrcnn_loss = roi_heads.fastrcnn_loss
original_focal_loss = torchvision.ops.sigmoid_focal_loss


def build_faster_rcnn(anchors=(32, 64, 128, 256, 512), proposals=1000, box_loss='smooth_l1'):
    roi_heads.fastrcnn_loss = giou_fastrcnn_loss if box_loss == 'giou' else original_fastrcnn_loss
    model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT', min_size=IMGSZ, max_size=2 * IMGSZ,
                                       rpn_post_nms_top_n_test=proposals,
                                       box_detections_per_img=MAX_DET, box_score_thresh=SCORE_THRESHOLD)
    model.rpn.anchor_generator = AnchorGenerator(tuple((size,) for size in anchors),
                                                 ((0.5, 1.0, 2.0),) * len(anchors))
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 9)
    return model


def build_retinanet(anchors=(32, 64, 128, 256, 512), alpha=0.25, gamma=2.0):
    retinanet.sigmoid_focal_loss = partial(original_focal_loss, alpha=alpha, gamma=gamma)
    sizes = tuple((x, int(x * 2 ** (1 / 3)), int(x * 2 ** (2 / 3))) for x in anchors)
    generator = AnchorGenerator(sizes, ((0.5, 1.0, 2.0),) * len(sizes))
    model = retinanet_resnet50_fpn_v2(weights='DEFAULT', min_size=IMGSZ, max_size=2 * IMGSZ,
                                      detections_per_img=MAX_DET, score_thresh=SCORE_THRESHOLD, topk_candidates=2000)
    model.anchor_generator = generator
    model.head.classification_head = RetinaNetClassificationHead(
        model.backbone.out_channels, generator.num_anchors_per_location()[0], 9,
        norm_layer=partial(nn.GroupNorm, 32))
    return model

In [ ]:
def train_torchvision(model, epochs=EPOCHS, lr=LR):
    torch.manual_seed(SEED)
    loader = DataLoader(FrameDataset(TRAIN_LIMIT), batch_size=BATCH, shuffle=True, collate_fn=collate,
                        num_workers=2 if DEVICE == 'cuda' else 0)
    model.to(DEVICE).train()
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                                lr=lr, momentum=0.9, weight_decay=5e-4)
    steps = epochs * len(loader)
    warmup = min(500, max(1, steps // 10))
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [
        torch.optim.lr_scheduler.LinearLR(optimizer, 0.01, total_iters=warmup),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, max(1, steps - warmup))], milestones=[warmup])
    scaler = torch.amp.GradScaler(enabled=DEVICE == 'cuda')

    for epoch in range(epochs):
        losses = []
        for images, targets in tqdm(loader, desc=f'epoch {epoch + 1}/{epochs}'):
            images = [image.to(DEVICE) for image in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
                loss = sum(model(images, targets).values())
            optimizer.zero_grad()
            if torch.isfinite(loss):
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                losses.append(loss.item())
            scheduler.step()
        print(f'epoch {epoch + 1}: loss {np.mean(losses):.3f}')


@torch.no_grad()
def torchvision_rows(model):
    model.to(DEVICE).eval()
    rows, seconds = [], 0.0
    for start in tqdm(range(0, len(val_frames), BATCH)):
        batch = val_frames[start:start + BATCH]
        images = [(read_image(str(path)).float() / 255).to(DEVICE) for _, path in batch]
        begin = time.time()
        with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
            outputs = model(images)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        seconds += time.time() - begin
        for (image_id, _), output in zip(batch, outputs):
            for (x1, y1, x2, y2), score, label in zip(output['boxes'].float().tolist(), output['scores'].tolist(),
                                                      output['labels'].tolist()):
                rows.append({'image_id': image_id, 'category_id': label, 'score': score,
                             'bbox': [x1, y1, x2 - x1, y2 - y1]})
    return rows, 1000 * seconds / len(val_frames)


def run_torchvision(name, family, settings, make_model, **train_args):
    if not should_run(name):
        return
    weights = RUNS / name / 'model.pt'
    if not weights.exists() and not TRAIN:
        print('Not trained yet, skipping:', name)
        return
    weights.parent.mkdir(parents=True, exist_ok=True)
    model = make_model()
    start = time.time()
    if weights.exists():
        model.load_state_dict(torch.load(weights, map_location='cpu'))
    else:
        train_torchvision(model, **train_args)
        torch.save(model.state_dict(), weights)
    minutes = (time.time() - start) / 60
    rows, ms = torchvision_rows(model)
    print(save_result(name, family, settings, {**coco_metrics(rows), 'ms_per_frame': ms, 'train_minutes': minutes}))
    del model
    torch.cuda.empty_cache()


## Anchor sizes from the data
Anchors must match box sizes after the model resizes the frame. This cell rescales train boxes the way the detector will, so anchors can be chosen from real sizes. The default smallest anchor is 32 px.

In [ ]:
train_coco = json.loads((DATA / 'train' / 'annotations_coco.json').read_text())
frame_size = {im['id']: (im['width'], im['height']) for im in train_coco['images']}
boxes = pd.DataFrame([(a['image_id'], a['bbox'][2], a['bbox'][3]) for a in train_coco['annotations'] if not a['iscrowd']],
                     columns=['image_id', 'w', 'h'])
size = boxes.image_id.map(frame_size)
scale = IMGSZ / size.map(min)
scale = np.minimum(scale, 2 * IMGSZ / size.map(max))
side = np.sqrt(boxes.w * boxes.h) * scale

print(side.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1))
print('boxes under 32 px after resizing:', f'{(side < 32).mean():.1%}')

## Experiments
Each family has a baseline and two experiments, changing one thing at a time. `SMALL_ANCHORS` follows the size table above.

Order matters when time is short: run every `base` first, then the experiments.

In [ ]:
SMALL_ANCHORS = (16, 32, 64, 128, 256)

In [ ]:
run_ultralytics('baseline-yolo11n-960', 'anchor-free', 'imgsz 960', lambda: YOLO('yolo11n.pt'))
run_torchvision('frcnn-base', 'two-stage', 'default anchors', build_faster_rcnn)
run_torchvision('retinanet-base', 'one-stage anchor', 'default anchors', build_retinanet)
run_ultralytics('rtdetr-base', 'query-based', 'imgsz 640, lr0 1e-4', lambda: RTDETR('rtdetr-l.pt'),
                imgsz=640, batch=8, epochs=EPOCHS, lr0=1e-4, optimizer='AdamW')

In [ ]:
run_torchvision('frcnn-anchors', 'two-stage', 'small anchors, 2000 proposals',
                lambda: build_faster_rcnn(anchors=SMALL_ANCHORS, proposals=2000))
run_torchvision('frcnn-giou', 'two-stage', 'small anchors, GIoU box loss',
                lambda: build_faster_rcnn(anchors=SMALL_ANCHORS, proposals=2000, box_loss='giou'))

In [ ]:
run_torchvision('retinanet-anchors', 'one-stage anchor', 'small anchors',
                lambda: build_retinanet(anchors=SMALL_ANCHORS))
run_torchvision('retinanet-focal', 'one-stage anchor', 'small anchors, focal gamma 1.5',
                lambda: build_retinanet(anchors=SMALL_ANCHORS, gamma=1.5))

In [ ]:
run_ultralytics('yolo11n-1280', 'anchor-free', 'imgsz 1280', lambda: YOLO('yolo11n.pt'), imgsz=1280, batch=4)
run_ultralytics('yolo11n-box', 'anchor-free', 'imgsz 960, box loss gain 10', lambda: YOLO('yolo11n.pt'), box=10.0)

In [ ]:
run_ultralytics('rtdetr-lr', 'query-based', 'imgsz 640, lr0 2e-4', lambda: RTDETR('rtdetr-l.pt'),
                imgsz=640, batch=8, epochs=EPOCHS, lr0=2e-4, optimizer='AdamW')
run_ultralytics('rtdetr-960', 'query-based', 'imgsz 960, lr0 1e-4', lambda: RTDETR('rtdetr-l.pt'),
                imgsz=960, batch=4, epochs=EPOCHS, lr0=1e-4, optimizer='AdamW')

## Comparison

In [ ]:
table = pd.read_csv(RESULTS).sort_values('mAP', ascending=False)
display(table.round(3))

order = table.sort_values('mAP')
fig, axes = plt.subplots(1, 3, figsize=(16, 0.42 * len(order) + 1.4), sharey=True)
for ax, col, title in zip(axes, ['mAP', 'AP_small', 'ms_per_frame'], ['mAP', 'AP_small', 'Inference, ms per frame']):
    ax.barh(order.run, order[col], color='#2a78d6')
    for y, value in enumerate(order[col]):
        ax.text(value, y, f' {value:.3f}' if col != 'ms_per_frame' else f' {value:.0f}', va='center', fontsize=9)
    ax.set_title(title, loc='left')
    ax.set_xticks([])
    ax.margins(x=0.18)
    ax.spines[['top', 'right', 'bottom']].set_visible(False)
plt.tight_layout()
plt.show()